# Vietnamese evidence embeddings — multilingual-e5-large

Notebook Kaggle này clone code từ GitHub, stream corpus chunk từ Hugging Face, tạo embedding trên GPU và upload từng Parquet shard sang dataset đầu ra.

Trước khi chạy: bật **Internet**, chọn **GPU** và tạo Kaggle secret `HF_TOKEN` có quyền ghi Hugging Face. `GITHUB_TOKEN` chỉ cần nếu GitHub repo là private.

In [ ]:
# Kết nối GitHub và lấy đúng phiên bản pipeline
import base64
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/mmmm144/Fact-checking.git"
GITHUB_BRANCH = "data"
REPO_DIR = Path("/kaggle/working/Fact-checking")

try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None

git_auth = []
if github_token:
    basic = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", *git_auth, "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=REPO_DIR, check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", *git_auth, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready at {REPO_DIR} (commit {commit})")

In [ ]:
# Cài dependencies tương thích với Kaggle
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "embedding/requirements-kaggle.txt")], check=True)
print("Dependencies installed")

In [ ]:
# Cấu hình job
import os
from kaggle_secrets import UserSecretsClient

HF_SOURCE_REPO = "Loctran123/vietnamese-evidence-corpus-chunked"
HF_SOURCE_REVISION = "3303a74a1ee6c23f231d9471d1eaa6d1a3009e12"
HF_OUTPUT_REPO = "Loctran123/vietnamese-evidence-corpus-embeddings-e5-large"
MODEL_ID = "intfloat/multilingual-e5-large"
BATCH_SIZE = 24       # giảm còn 16 hoặc 8 nếu thiếu VRAM
ROWS_PER_SHARD = 5000
PRIVATE_OUTPUT = False

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
assert HF_SOURCE_REPO != HF_OUTPUT_REPO, "Output repo must be separate from source chunks"
print("Configuration ready; HF token loaded without displaying it.")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật.")

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)

print("GPU:", gpu_name)
print("CUDA:", torch.version.cuda)
print("CUDA capability:", capability)

if capability < (7, 0):
    raise RuntimeError(
        f"{gpu_name} có capability {capability}, không tương thích với "
        "PyTorch hiện tại. Hãy chuyển Kaggle Accelerator sang T4."
    )

# Kiểm tra thực sự có chạy phép tính trên GPU
x = torch.ones(1, device="cuda")
print("CUDA tensor test:", x)

In [ ]:
# Chạy embedding. Script tự resume từ shard HF kế tiếp nếu session bị ngắt.
command = [
    sys.executable,
    str(REPO_DIR / "embedding/embed_e5_kaggle.py"),
    "--source-repo", HF_SOURCE_REPO,
    "--source-revision", HF_SOURCE_REVISION,
    "--output-repo", HF_OUTPUT_REPO,
    "--model-id", MODEL_ID,
    "--batch-size", str(BATCH_SIZE),
    "--rows-per-shard", str(ROWS_PER_SHARD),
    "--output-dir", "/kaggle/working/e5_embeddings",
]
if PRIVATE_OUTPUT:
    command.append("--private-output")
subprocess.run(command, check=True)

## Sau khi hoàn tất

Dataset đầu ra có `manifest.json` với trạng thái `complete`. Khi retrieval, embedding claim bằng prefix `query: `, chuẩn hóa L2, rồi dùng cosine similarity hoặc inner product với cột `embedding`. Không commit các file embedding lớn vào GitHub.